# Richardson Attention (Superconducting Attention): Interactive Colab Demo

**Eliminating the Multi-Hop Context Horizon in Pre-Trained Transformers via Dual-State Boltzmann Extrapolation and Renormalization Relays**

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.23445174.svg)](https://doi.org/10.5281/zenodo.23445174) [![GitHub Repository](https://img.shields.io/badge/GitHub-tryggth%2Frichardson--attention-181717?logo=github)](https://github.com/tryggth/richardson-attention)

* **CERN Zenodo Archival Preprint:** [https://doi.org/10.5281/zenodo.23445174](https://doi.org/10.5281/zenodo.23445174)
* **GitHub Repository & Code:** [https://github.com/tryggth/richardson-attention](https://github.com/tryggth/richardson-attention)

---

### Overview: The Five Core Mathematical Laws of Richardson Attention

Modern hybrid-attention Transformers (`google/gemma-4-E2B-it` and `google/gemma-4-E4B-it`) interleave localized **Sliding Window Attention (SWA)** layers ($W = 512$ tokens) with sparse **Full Global Attention** layers (`L5, L11, L17, L23, L29, L35, L41` in `gemma-4-E4B-it`). While they achieve near-$100\%$ accuracy on single-hop passkey retrieval, they suffer a **catastrophic phase transition on multi-hop compositional pointer chasing** (`Director -> Sector -> Vault -> Genesis Code`) when intermediate hops are separated by barriers exceeding the 512-token SWA window and followed by adversarial recency decoys.

**Richardson Attention** restores **1-Hop through 5-Hop compositional reasoning** across 8K-12K contexts with **zero fine-tuning** and **zero perplexity degradation** via five coupled mechanisms:

1. **Stage 1 Precision-Matrix Brain Scan ($\mathbf{P} = \mathbf{R}^+$):** Measures each head's effective attention decay mass $\hat{\alpha}_{l,h} = (\bar{d}_{l,h} + \epsilon)^{-1}$ and inter-head entropy Variance Inflation Factor $\text{VIF}_{l,h} = (\mathbf{R}^+)_{m,m}$ to isolate a sparse **Global Logic Quorum** ($Q \in \{14, 28, 42\}$ heads out of 336) while leaving $91.7\%\text{--}95.8\%$ of heads untouched.
2. **Write/Read Asymmetric Query-Slice Gating ($W_q = 64$):** Leaves document prefill rows ($i < T - W_q$) under unmodified standard softmax to preserve intra-clause KV-cache entity binding (`SECTOR-KRONOS <-> VAULT-LAZARUS`), activating superconducting extrapolation strictly on the active query readout slice ($i \in [T - W_q, T)$) with `<bos>` sink pinning ($d_{i,0} = 0$) and a 32-token local syntax guardband ($d_{\text{long}} = \max(0, i - j - 32)$).
3. **Iso-Potential Cubic Superconducting Plateau ($\Phi_l(i,j)$):** Replaces concave logarithmic tilts (which boost mid-context decoys at $87\%$ of target strength) with a convex cubic ramp that saturates at a flat $\Phi = 1.0$ plateau across all upstream registry zones ($d \ge c_l T$):
   $$\Phi_l(i, j) = \min\!\left(1.0,\; \frac{\max(0,\, i - j - 32)}{c_l \cdot \max(T, L_{\text{calib}})}\right)^3$$
4. **Dual-State Symmetric Boltzmann Extrapolation:** Evaluates attention at symmetric positive and negative potential shifts $S_{\text{free}} = \text{softmax}(L + \Delta L)$ and $S_{\text{heavy}} = \text{softmax}(L - \Delta L)$ and forms the Richardson-extrapolated distribution:
   $$S_{\text{super}} = \mathcal{N}\!\Big(\max\big(0,\; (1 + \eta) S_{\text{free}} - \eta S_{\text{heavy}}\big)\Big)$$
5. **Depth-Progressive Renormalization Group (RG) Relays & SWA Decoy Shielding:** Steps the plateau cutoff $c_l$ backward across the 7 global layers ($c_l \in [0.52, 0.68, 0.85]$ for 3-Hop; $c_l \in [0.38, 0.44, 0.52, 0.64, 0.74, 0.85, 0.90]$ for 5-Hop) while optionally damping the $48 < d \le 512$ tail inside SWA layers ($\beta_{\text{swa}} = 2.5$).

---

### How This Notebook is Organized
* **Part 1 (Cells 2-3) — Instant Physics & Telemetry Explorer (No GPU or Token Required):** Runs in **3 seconds on any CPU/GPU runtime** to plot the potential landscapes, Phase 5 (3-Hop at 8,270 tokens) & Phase 6 (1-to-5 Hop at 9,912 tokens) benchmark curves, and the 6-Gap Residual Fiber-Bundle Telemetry.
* **Part 2 (Cells 4-6) — Live Gemma-4 GPU Inference & Brain Scan:** Automatically detects your GPU VRAM (`google/gemma-4-E4B-it` on L4/A100 >= 20GB VRAM, or `google/gemma-4-E2B-it` on Free Colab T4 15GB VRAM), executes the live Stage 1 Precision-Matrix Brain Scan, and runs a side-by-side **Before vs. After** multi-hop pointer-chasing evaluation.

In [ ]:
# 1. Clone the repository and install richardson-attention
import os
if not os.path.exists("richardson-attention"):
    !git clone https://github.com/tryggth/richardson-attention.git
%cd richardson-attention
!pip install -q -e .
!pip install -q -U "transformers>=5.5.0" accelerate matplotlib

In [ ]:
# Part 1: Instant Interactive Physics & Phase 5/6 Telemetry Explorer (No GPU Required)
import json
import numpy as np
import matplotlib.pyplot as plt

with open("results/richardson_phase5_results.json", "r") as f:
    p5_data = json.load(f)
with open("results/richardson_phase6_vertical_results.json", "r") as f:
    p6_data = json.load(f)
with open("results/richardson_phase7_standard_benchmarks.json", "r") as f:
    p7_data = json.load(f)
with open("results/richardson_phase8_fiber_bundle_results.json", "r") as f:
    p8_data = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(19, 5.4), dpi=120)

# --- Panel 1: Potential Geometry ---
ax1 = axes[0]
T = 8270.0
d = np.linspace(0, T, 1000)
d_long = np.maximum(0.0, d - 32.0)
phi_linear = d_long / T
phi_log = np.log1p(d_long) / np.log1p(T)
phi_cubic = np.minimum(1.0, d_long / (0.54 * T)) ** 3.0

ax1.plot(d / T, phi_log, "--", color="#d95f02", lw=2.0, label="Concave Log (Boosts Decoys: 0.87)")
ax1.plot(d / T, phi_linear, ":", color="#7570b3", lw=2.0, label="Linear Ramp (Starves Zone C: 0.60)")
ax1.plot(d / T, phi_cubic, "-", color="#1b9e77", lw=3.0, label="Cubic Plateau (c=0.54, p=3)")
ax1.axvspan(0.0, 0.50, color="#ffcccc", alpha=0.25, label="Adversarial Decoy Region (d/T < 0.50)")
for z_frac, z_lbl in [(0.60, "Zone C"), (0.79, "Zone B"), (0.99, "Zone A")]:
    ax1.axvline(z_frac, color="#2b5c8f", linestyle="-.", alpha=0.6)
    ax1.text(z_frac - 0.02, 0.52, z_lbl, rotation=90, fontsize=9, fontweight="bold", color="#2b5c8f")
ax1.set_title("1. Iso-Potential Cubic Superconducting Plateau", fontsize=11, fontweight="bold")
ax1.set_xlabel("Normalized Causal Distance (d / T)")
ax1.set_ylabel("Potential Amplitude Φ(d)")
ax1.set_ylim(-0.05, 1.12)
ax1.grid(True, alpha=0.25)
ax1.legend(loc="lower right", fontsize=8)

# --- Panel 2: Phase 5 3-Hop Benchmark (8,270 Tokens) ---
ax2 = axes[1]
p5_configs = [
    ("1. Baseline (Unmodified)", "Baseline\n(Unmodified)"),
    ("3. Query-Slice Dual-Log (Q28, Wq=64, b=3.0)", "Query-Slice\nDual-Log (Q28)"),
    ("4. Query-Slice Cubic-Plateau (Q14, Unif c=0.54, b=3.5)", "Cubic Plateau\n(Q14, c=0.54)"),
    ("8. Deep-Quorum RG Relay (Q42, c=[.52,.68,.85], b=4.0)", "Deep-Quorum RG\n(Q42, Stepped)"),
]
p5_lookup = {(row["configuration"], row["hop_task"]): row for row in p5_data["evaluations"]}
x = np.arange(len(p5_configs))
w = 0.26
clean_vals = [p5_lookup[(c, "3-Hop (Clean-6d)")]["target_prob_pct"] for c, _ in p5_configs]
heavy_vals = [p5_lookup[(c, "3-Hop (Heavy-16d)")]["target_prob_pct"] for c, _ in p5_configs]
decoy_vals = [p5_lookup[(c, "3-Hop (Heavy-16d)")]["decoy_omega_prob_pct"] for c, _ in p5_configs]

ax2.bar(x - w, clean_vals, width=w, color="#2b83ba", label="3-Hop Clean-6d Target (79421)")
ax2.bar(x, heavy_vals, width=w, color="#1b9e77", label="3-Hop Heavy-16d Target (79421)")
ax2.bar(x + w, decoy_vals, width=w, color="#d7191c", label="Heavy-16d Decoy Error (63819)")
ax2.set_xticks(x)
ax2.set_xticklabels([lbl for _, lbl in p5_configs], fontsize=8.5)
ax2.set_ylim(0, 112)
ax2.set_ylabel("Probability (%)")
ax2.set_title("2. Phase 5: 3-Hop Pointer Chasing (8,270 Tokens)", fontsize=11, fontweight="bold")
ax2.grid(True, axis="y", alpha=0.25)
ax2.legend(loc="upper right", fontsize=8)

# --- Panel 3: Phase 6 1-to-5 Hop Scaling (9,912 Tokens) ---
ax3 = axes[2]
hops = ["1-Hop", "2-Hop", "3-Hop", "4-Hop", "5-Hop"]
p6_lookup = {(row["configuration"], row["hop_task"]): row for row in p6_data["evaluations"]}
p6_lines = [
    ("1. Baseline (Unmodified)", "Baseline (Unmodified)", "o--", "#d7191c"),
    ("3. Horiz-Only Cubic (Q14, c=0.40, b=3.5)", "Uniform Cubic (Q14, c=0.40)", "s-.", "#2b83ba"),
    ("7. 2D Cubic Q14 + SWA Decoy Shield (sw=2.5)", "Cubic Q14 + SWA Shield (sw=2.5)", "^-", "#fdae61"),
    ("4. Horiz-Only Stepped RG (Q28, RG, b=3.5)", "7-Step RG Relay (Q28, c=[.38..90])", "D-", "#1b9e77"),
]
for cfg_key, lbl, style, col in p6_lines:
    y = [p6_lookup[(cfg_key, h)]["target_prob_pct"] for h in hops]
    ax3.plot(hops, y, style, color=col, lw=2.5, markersize=6, label=lbl)
ax3.set_ylim(-3, 106)
ax3.set_ylabel("Target (79421) Probability (%)")
ax3.set_title("3. Phase 6: 1-to-5 Hop Scaling (9,912 Tokens)", fontsize=11, fontweight="bold")
ax3.grid(True, alpha=0.25)
ax3.legend(loc="lower left", fontsize=8)

plt.tight_layout()
plt.show()

# Print 6-Gap Residual Fiber-Bundle Telemetry Table from Phase 6 (5-Hop)
base_5h = p6_lookup[("1. Baseline (Unmodified)", "5-Hop")]["vertical_gap_telemetry"]
rg_5h = p6_lookup[("4. Horiz-Only Stepped RG (Q28, RG, b=3.5)", "5-Hop")]["vertical_gap_telemetry"]

print("=" * 105)
print("PHASE 6 RESIDUAL FIBER-BUNDLE TELEMETRY ACROSS 5-LAYER SWA+MLP BLOCKS (5-HOP AT 9,912 TOKENS)")
print("=" * 105)
print(f"{'5-Layer SWA Gap':<15} | {'||u_global||_2':<14} | {'||dH_SWA||_2':<13} | {'Drift/Signal':<12} | {'RMS Infl':<9} | {'Base Cosine':<12} | {'Stepped RG Cosine'}")
print("-" * 105)
for gap in ["L5->L11", "L11->L17", "L17->L23", "L23->L29", "L29->L35", "L35->L41"]:
    b = base_5h[gap]
    r = rg_5h[gap]
    tag = "  <-- Phase Boundary!" if gap == "L23->L29" else ("  <-- Reversed (+)!" if gap == "L35->L41" else "")
    print(
        f"{gap:<15} | {b['global_attn_u_norm']:<14.4f} | {b['swa_5layer_drift_norm']:<13.4f} | "
        f"{b['drift_to_signal_ratio']:<12.4f} | {b['rms_norm_inflation']:<9.4f} | "
        f"{b['cosine_retention']:+12.4f} | {r['cosine_retention']:+10.4f}{tag}"
    )
print("=" * 105)

# Print Phase 7 Multi-Seed Standard Benchmark Tables (RULER VT, BABILong qa3, & Lost in the Middle)
print("\n" + "=" * 105)
print("PHASE 7A: MULTI-SEED NVIDIA RULER `VT` & BABILONG `qa3` SUMMARY (N=10 RANDOMIZED SEEDS EACH)")
print("=" * 105)
print(f"{'Benchmark Suite':<20} | {'Configuration':<40} | {'Top-1 Acc':<10} | {'Mean Target Prob':<19} | {'Mean Decoy Err'}")
print("-" * 105)
for suite_lbl, key in [("NVIDIA RULER VT", "ruler_vt_summary"), ("BABILong qa3", "babilong_qa3_summary")]:
    for row in p7_data[key]:
        print(
            f"{suite_lbl:<20} | {row['configuration']:<40} | {row['top1_accuracy_pct']:>7.1f}%  | "
            f"{row['mean_target_prob_pct']:>7.4f}% +/- {row['std_target_prob_pct']:<6.2f}% | {row['mean_decoy_prob_pct']:>8.4f}%"
        )
    print("-" * 105)

print("\n" + "=" * 105)
print("PHASE 7B: 'LOST IN THE MIDDLE' MULTI-DOCUMENT DEPTH SWEEP (15% TO 75% DOCUMENT DEPTH)")
print("=" * 105)
print(f"{'Configuration':<42} | {'15% Depth':<11} | {'30% Depth':<11} | {'45% Depth':<11} | {'60% Depth':<11} | {'75% Depth'}")
print("-" * 105)
for row in p7_data["lost_in_the_middle_sweep"]:
    d_map = row["depth_target_prob_pct"]
    print(
        f"{row['configuration']:<42} | {d_map['15%']:>9.4f}% | {d_map['30%']:>9.4f}% | "
        f"{d_map['45%']:>9.4f}% | {d_map['60%']:>9.4f}% | {d_map['75%']:>9.4f}%"
    )
print("=" * 105)

# Print Phase 8 Fiber-Bundle Procrustes Diagnostics & Vocabulary-Fiber Pullback Summary
print("\n" + "=" * 105)
print("PHASE 8A: FIBER-BUNDLE PROCRUSTES SVD DIAGNOSTICS ACROSS THE 6 SWA GAPS (d=2,560)")
print("=" * 105)
print(f"{'SWA Layer Gap':<15} | {'Unrotated Cos':<15} | {'Procrustes (r=128) Cos':<24} | {'Top-16 SVD':<12} | {'Top-128 SVD':<13} | {'Mean ||u_l||_2'}")
print("-" * 105)
for gap_k, diag in p8_data["fiber_procrustes_diagnostics"].items():
    print(
        f"{gap_k:<15} | {diag['raw_unrotated_cosine']:>+13.4f}  | "
        f"{diag['procrustes_rank128_cosine']:>+21.4f}   | "
        f"{diag['top16_singular_energy_pct']:>9.2f}%  | "
        f"{diag['top128_singular_energy_pct']:>10.2f}%  | {diag['mean_calib_u_norm']:>10.2f}"
    )
print("-" * 105)

print("\n" + "=" * 105)
print("PHASE 8B: VOCABULARY-FIBER GAUGE PULLBACK VS. PROCRUSTES SKIP HIGHWAY (5-HOP & BABILONG QA3)")
print("=" * 105)
print(f"{'Configuration':<58} | {'5-Hop Prob':<11} | {'5-Hop Decoy':<11} | {'BABILong Prob':<15} | {'BABILong Decoy'}")
print("-" * 105)
for row in p8_data["evaluations"]:
    print(
        f"{row['configuration']:<58} | {row['five_hop_target_prob_pct']:>8.4f}%  | "
        f"{row['five_hop_decoy_prob_pct']:>9.4f}%  | {row['babilong_qa3_mean_target_prob_pct']:>11.4f}%  | "
        f"{row['babilong_qa3_mean_decoy_prob_pct']:>9.4f}%"
    )
print("=" * 105)

In [ ]:
# Part 2: GPU Auto-Detection & Gemma 4 Model Loader
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), (
    "Part 2 requires a CUDA GPU runtime. In Google Colab, select Runtime -> Change runtime type -> T4 / L4 / A100 GPU."
)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
device_name = torch.cuda.get_device_name(0)

# Select E4B on >=20GB GPUs (L4/A100) or E2B on <20GB GPUs (Free Colab T4 15GB) so bfloat16 fits without OOM
if vram_gb >= 20.0:
    MODEL_ID = "google/gemma-4-E4B-it"
    DEMO_TARGET_TOKENS = 8000
else:
    MODEL_ID = "google/gemma-4-E2B-it"
    DEMO_TARGET_TOKENS = 4500

print(f"Detected GPU: {device_name} ({vram_gb:.1f} GB VRAM) -> Selected {MODEL_ID} (Target Context: ~{DEMO_TARGET_TOKENS} tokens)")

# Authenticate with HuggingFace if needed for gated Gemma-4 weights
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    from huggingface_hub import notebook_login
    notebook_login()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
    token=hf_token,
).eval()
print(f"Successfully loaded {MODEL_ID} in bfloat16 with eager attention.")

In [ ]:
# Live Stage 1 Precision-Matrix Brain Scan (P = R^+) & Quorum Isolation
import matplotlib.pyplot as plt
from richardson_attention import attach_richardson_attention
from richardson_attention.controller import get_text_config_and_layers

controller = attach_richardson_attention(
    model,
    tokenizer,
    heads_per_layer=2,
    mode="dual_cubic_plateau",
    beta=3.5,
    eta=0.75,
    swa_shield_beta=2.5,
)

_, layers = get_text_config_and_layers(model)
print("=" * 78)
print(f"ISOLATED GLOBAL LOGIC QUORUM ({len(controller.target_heads)} HEADS SELECTED ACROSS {len(controller.layer_horizons)} GLOBAL LAYERS)")
print("=" * 78)
print(f"{'Layer':<8} | {'Head':<6} | {'Effective Mass (alpha)':<24} | {'Calibrated Coupling (kappa)':<28} | {'Horizon c_l'}")
print("-" * 78)
for (l_idx, h_idx), kappa in sorted(controller.target_heads.items()):
    mass = controller.head_masses[(l_idx, h_idx)]
    c_l = controller.layer_horizons.get(l_idx, 0.54)
    print(f"L{l_idx:<7} | H{h_idx:<5} | {mass:<24.6f} | {kappa:<28.4f} | {c_l:.2f}")
print("=" * 78)

# Plot Effective Mass distribution across all layers (Sliding Window vs Full Global Attention)
fig, ax = plt.subplots(figsize=(12, 4.2), dpi=120)
swa_x, swa_y = [], []
glob_x, glob_y = [], []
quorum_x, quorum_y = [], []

for (l_idx, h_idx), mass in controller.head_masses.items():
    ltype = layers[l_idx].self_attn.layer_type_str
    if (l_idx, h_idx) in controller.target_heads:
        quorum_x.append(l_idx)
        quorum_y.append(mass)
    elif ltype == "full_attention":
        glob_x.append(l_idx)
        glob_y.append(mass)
    else:
        swa_x.append(l_idx)
        swa_y.append(mass)

ax.scatter(swa_x, swa_y, alpha=0.45, s=22, color="#d95f02", label="Sliding-Window Attention Heads (High Mass / Short Horizon)")
ax.scatter(glob_x, glob_y, alpha=0.65, s=30, color="#7570b3", label="Unmodified Global Attention Heads")
ax.scatter(quorum_x, quorum_y, alpha=1.0, s=65, color="#1b9e77", edgecolors="black", zorder=5, label="Selected Global Logic Quorum (Richardson Active)")
ax.set_yscale("log")
ax.set_xlabel("Transformer Layer Index (l)")
ax.set_ylabel("Effective Attention Decay Mass α (log scale)")
ax.set_title(f"Live Stage 1 Brain Scan on {MODEL_ID}: Homunculus Mass Spectrum", fontweight="bold")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper right", fontsize=8.5)
plt.tight_layout()
plt.show()

In [ ]:
# Live "Before & After" Multi-Hop Pointer Chasing Evaluation (1-Hop, 2-Hop, 3-Hop)
import torch
import numpy as np
import matplotlib.pyplot as plt

def build_live_multihop_prompts(target_tokens: int = 8000):
    scale = max(1, int(round(target_tokens / 8000.0 * 40)))
    decoy_reps = max(6, int(round(target_tokens / 8000.0 * 16)))
    filler_step = max(3, int(round(target_tokens / 8000.0 * 7)))

    zone_a = (
        "=== ZONE A: TIER-0 IMMUTABLE VAULT CODE REGISTRY ===\n"
        "- VAULT-ALPHA    | TIER-0 GENESIS CODE: 49281\n"
        "- VAULT-HELIOS   | TIER-0 GENESIS CODE: 54712\n"
        "- VAULT-HYPERION | TIER-0 GENESIS CODE: 83914\n"
        "- VAULT-LAZARUS  | TIER-0 GENESIS CODE: 79421\n"
        "- VAULT-NYX      | TIER-0 GENESIS CODE: 37192\n"
        "- VAULT-OMEGA    | TIER-0 GENESIS CODE: 63819\n"
        "- VAULT-VULCAN   | TIER-0 GENESIS CODE: 28491\n"
        "- VAULT-ZEPHYR   | TIER-0 GENESIS CODE: 91824\n\n"
    )
    zone_b = (
        "\n=== ZONE B: TIER-0 COMMAND SECTOR ROUTING REGISTRY ===\n"
        "- SECTOR-ATLAS    | ASSIGNED PRIMARY VAULT: VAULT-OMEGA\n"
        "- SECTOR-BOREALIS | ASSIGNED PRIMARY VAULT: VAULT-ALPHA\n"
        "- SECTOR-CYGNUS   | ASSIGNED PRIMARY VAULT: VAULT-ZEPHYR\n"
        "- SECTOR-DRACO    | ASSIGNED PRIMARY VAULT: VAULT-HELIOS\n"
        "- SECTOR-EREBUS   | ASSIGNED PRIMARY VAULT: VAULT-VULCAN\n"
        "- SECTOR-KRONOS   | ASSIGNED PRIMARY VAULT: VAULT-LAZARUS\n"
        "- SECTOR-ORION    | ASSIGNED PRIMARY VAULT: VAULT-NYX\n"
        "- SECTOR-PEGASUS  | ASSIGNED PRIMARY VAULT: VAULT-HYPERION\n\n"
    )
    zone_c = (
        "\n=== ZONE C: TIER-0 DIRECTORATE COMMAND REGISTRY ===\n"
        "- DIRECTOR-CHEN     | AUTHORIZED COMMAND SECTOR: SECTOR-BOREALIS\n"
        "- DIRECTOR-KAEL     | AUTHORIZED COMMAND SECTOR: SECTOR-ORION\n"
        "- DIRECTOR-MERCER   | AUTHORIZED COMMAND SECTOR: SECTOR-DRACO\n"
        "- DIRECTOR-ROSTOVA  | AUTHORIZED COMMAND SECTOR: SECTOR-PEGASUS\n"
        "- DIRECTOR-STERLING | AUTHORIZED COMMAND SECTOR: SECTOR-ATLAS\n"
        "- DIRECTOR-THORNE   | AUTHORIZED COMMAND SECTOR: SECTOR-CYGNUS\n"
        "- DIRECTOR-VANCE    | AUTHORIZED COMMAND SECTOR: SECTOR-KRONOS\n"
        "- DIRECTOR-WEBB     | AUTHORIZED COMMAND SECTOR: SECTOR-EREBUS\n\n"
    )
    filler = (
        "Telemetry Log: Cryogenic bus voltage nominal at 4.2K; Josephson junction phase coherence locked. "
        "Syndrome parity check completed with zero faults; spectral damper calibration steady. "
    )
    decoy = (
        "\n[TIER-2 RUNTIME AUDIT] Temporary cross-check: DIRECTOR-VANCE inspected SECTOR-ATLAS and "
        "VAULT-OMEGA using temporary maintenance override key 11111 (non-binding).\n"
    )
    prefix = zone_a + (filler * scale) + zone_b + (filler * scale) + zone_c + (filler * filler_step + decoy) * decoy_reps + (filler * 4)

    return {
        "1-Hop": prefix + "\nQuestion: What is the 5-digit TIER-0 GENESIS CODE for VAULT-LAZARUS?\nAnswer: The 5-digit TIER-0 GENESIS CODE is ",
        "2-Hop": prefix + "\nQuestion: Trace the Sector Routing Registry to the Vault Code Registry: what is the 5-digit TIER-0 GENESIS CODE for the primary vault assigned to SECTOR-KRONOS?\nAnswer: The 5-digit TIER-0 GENESIS CODE is ",
        "3-Hop": prefix + "\nQuestion: Trace the Directorate Command Registry to the Sector Routing Registry and Vault Code Registry: what is the 5-digit TIER-0 GENESIS CODE for the primary vault commanded by DIRECTOR-VANCE?\nAnswer: The 5-digit TIER-0 GENESIS CODE is ",
    }

bos = tokenizer.bos_token or ""
device = next(model.parameters()).device
prompts = build_live_multihop_prompts(DEMO_TARGET_TOKENS)

target_tok = tokenizer.encode("79421", add_special_tokens=False)[0]
omega_tok = tokenizer.encode("63819", add_special_tokens=False)[0]
competing_codes = ["49281", "54712", "83914", "37192", "63819", "28491", "91824"]
all_vault_toks = [target_tok] + [tokenizer.encode(c, add_special_tokens=False)[0] for c in competing_codes]

live_results = {}
for cfg_name, mode, swa_beta in [
    ("1. Unmodified Baseline", "baseline", 0.0),
    ("2. Superconducting Richardson Attention", "dual_cubic_plateau", 2.5),
]:
    controller.mode = mode
    controller.swa_shield_beta = swa_beta
    live_results[cfg_name] = {}
    for hop_name, prompt_text in prompts.items():
        inp = tokenizer(bos + prompt_text, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model(**inp, use_cache=False, logits_to_keep=1)
            probs = torch.softmax(out.logits[0, -1, :].float(), dim=-1)
            t_prob = probs[target_tok].item() * 100.0
            d_prob = probs[omega_tok].item() * 100.0
            vault_probs = [probs[tok].item() for tok in all_vault_toks]
            v_rank = sorted(vault_probs, reverse=True).index(vault_probs[0]) + 1
        live_results[cfg_name][hop_name] = {
            "tokens": int(inp.input_ids.shape[1]),
            "target_pct": t_prob,
            "decoy_pct": d_prob,
            "rank": v_rank,
        }
        del inp, out

seq_tokens = live_results["1. Unmodified Baseline"]["3-Hop"]["tokens"]
print("=" * 96)
print(f"LIVE MULTI-HOP POINTER CHASING ON {MODEL_ID} ({seq_tokens} TOKENS)")
print("=" * 96)
print(f"{'Configuration':<40} | {'Hop':<7} | {'Target (79421)':<16} | {'Decoy (63819)':<15} | {'Vault Rank'}")
print("-" * 96)
for cfg_name, hop_dict in live_results.items():
    for hop_name, m in hop_dict.items():
        status = "#1 (WIN)" if m["rank"] == 1 else f"#{m['rank']} (FAIL)"
        print(f"{cfg_name:<40} | {hop_name:<7} | {m['target_pct']:>14.4f}% | {m['decoy_pct']:>13.4f}% | {status}")
    print("-" * 96)

# Plot Before & After Bar Chart
fig, ax = plt.subplots(figsize=(8.5, 4.5), dpi=120)
hops = ["1-Hop", "2-Hop", "3-Hop"]
x = np.arange(len(hops))
w = 0.34
base_t = [live_results["1. Unmodified Baseline"][h]["target_pct"] for h in hops]
rich_t = [live_results["2. Superconducting Richardson Attention"][h]["target_pct"] for h in hops]
ax.bar(x - w/2, base_t, width=w, color="#d7191c", label="1. Unmodified Baseline - Target (79421)")
ax.bar(x + w/2, rich_t, width=w, color="#1b9e77", label="2. Richardson Attention - Target (79421)")
ax.set_xticks(x)
ax.set_xticklabels(hops, fontweight="bold")
ax.set_ylim(0, 108)
ax.set_ylabel("Target (79421) Probability (%)")
ax.set_title(f"Live Before & After Multi-Hop Retrieval on {MODEL_ID} ({seq_tokens} Tokens)", fontweight="bold")
ax.grid(True, axis="y", alpha=0.25)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()